# DESC ELAsTiCC2 — SALT2 light-curve fitting with sncosmo

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-07

## Purpose

This notebook fits SNIa light curves from the ELAsTiCC2 training sample using the
**SALT2** model (`salt2-extended`) provided by `sncosmo`.  
The five SALT2 parameters — `z`, `t0`, `x0`, `x1`, `c` — are fitted **simultaneously
across all six LSST bands** (u, g, r, i, z, y) without any per-band renormalisation.

The SALT2 model encodes the full multi-band spectral energy distribution of SNIa;
no colour factor is needed between bands.

### References
- sncosmo documentation: https://sncosmo.readthedocs.io/en/stable/index.html
- ELAsTiCC2 dataset: DESC TD public data
- Notebook `01_readsnana/03_elasticc2_fit_lightcurves.ipynb` — data-loading pattern
- Notebook `02_sncosmo/01_sncosmo.ipynb` — sncosmo model usage


## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings

import numpy as np
import pandas as pd
import astropy.table
import matplotlib
from matplotlib import pyplot as plt

import sncosmo

# ── local library ──────────────────────────────────────────────────────────────
libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401
    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

## 1 · Parameters

In [ ]:
# ── Data selection ─────────────────────────────────────────────────────────────
OBJ_CLASS      = 'SNIa-SALT3'   # SNANA class label
Z_MIN          = 0.1            # redshift lower bound
Z_MAX          = 0.5            # redshift upper bound
FILE_NUM       = 1              # PHOT file index (1–40; None = all)
MIN_DETECTIONS = 8              # minimum detected points per object
DETECTED_ONLY  = True           # use only detected points (PHOTFLAG & photflag_detect)
N_CURVES       = 100             # number of events to fit and display
RANDOM_SEED    = 42

# ── SALT2 model ────────────────────────────────────────────────────────────────
SALT2_SOURCE   = 'salt2-extended'   # sncosmo source name
ZP             = 31.4               # zero-point used in SNANA / ELAsTiCC2  (AB system)
ZPSYS          = 'ab'

# LSST band names as understood by sncosmo (prefix 'lsst' + letter)
BANDS          = ['u', 'g', 'r', 'i', 'z', 'y']   # lower-case, as in sncosmo
BAND_PREFIX    = 'lsst'

# ── Data path ──────────────────────────────────────────────────────────────────
DATA_DIR   = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX = "ELASTICC2_TRAIN_02_"

# ── Plot colours per band (consistent with 03_elasticc2_fit_lightcurves) ───────
BAND_COLORS = {
    'u': '#cc0ccc',
    'g': '#00cc44',
    'r': '#cc0000',
    'i': '#ff4400',
    'z': '#886600',
    'y': '#442200'
}

# ── Display ────────────────────────────────────────────────────────────────────
NCOLS = 4

rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"SALT2 source : {SALT2_SOURCE}")
print(f"ZP={ZP}  zpsys={ZPSYS}")
print(f"Bands : {BANDS}")
print(f"N_CURVES={N_CURVES}")

## 2 · sncosmo model and band registration check

In [ ]:
# Load SALT2 model — sncosmo will download it on first use
salt2_model = sncosmo.Model(source=SALT2_SOURCE)
print(f"SALT2 model loaded: {salt2_model}")
print(f"Parameters: {salt2_model.param_names}")

# Verify that LSST band names are registered in sncosmo
print("\nChecking LSST band registration in sncosmo:")
for b in BANDS:
    bname = BAND_PREFIX + b
    try:
        band_obj = sncosmo.get_bandpass(bname)
        print(f"  {bname:10s}  →  {band_obj}  ✓")
    except Exception as e:
        print(f"  {bname:10s}  →  NOT FOUND: {e}")

## 3 · Load ELAsTiCC2 data

Same loading pattern as `01_readsnana/03_elasticc2_fit_lightcurves.ipynb`.

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)

_logger.info(f"Loading HEAD for {OBJ_CLASS}...")
head  = esr.get_head(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading truth for {OBJ_CLASS}...")
truth = esr.get_object_truth(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading light curves (file_num={FILE_NUM})...")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format='pandas')
_logger.info("Done.")

print(f"{all_ltcvs['SNID'].nunique()} objects loaded.")
print(f"Columns: {list(all_ltcvs.columns)}")

In [ ]:
# ── Filter: redshift range + minimum detections ───────────────────────────────
detcounts = (
    all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
    .groupby('SNID').agg('count')['MJD']
    .reset_index()
    .rename({'MJD': 'ndetect'}, axis=1)
)

truth_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')
subset = truth_counts[
    (truth_counts['ZCMB'] >= Z_MIN) &
    (truth_counts['ZCMB'] <  Z_MAX) &
    (truth_counts['ndetect'] >= MIN_DETECTIONS)
].copy()

print(f"{len(subset)} objects pass selection (z∈[{Z_MIN},{Z_MAX}), ndet≥{MIN_DETECTIONS}).")

## 4 · Helper: build an `astropy.Table` for sncosmo

`sncosmo.fit_lc` expects an `astropy.Table` with columns
`time`, `band`, `flux`, `fluxerr`, `zp`, `zpsys`.

The ELAsTiCC2 data use `BAND` values like `'r'`; we prepend `'lsst'` to get `'lsstr'`
as required by sncosmo.  
Note: ELAsTiCC2 uses capital `'Y'` for the y-band — we convert to lower-case `'y'`.

In [ ]:
def make_sncosmo_table(ltcv_df: pd.DataFrame,
                       detected_only: bool = True,
                       photflag_detect: int = None,
                       zp: float = ZP,
                       zpsys: str = ZPSYS,
                       band_prefix: str = BAND_PREFIX) -> astropy.table.Table:
    """Convert an ELAsTiCC2 light-curve DataFrame to an astropy.Table
    suitable for sncosmo.fit_lc.

    Parameters
    ----------
    ltcv_df         : DataFrame for a single SNID (columns MJD, FLUXCAL,
                      FLUXCALERR, BAND, PHOTFLAG)
    detected_only   : if True, keep only rows where PHOTFLAG & photflag_detect != 0
    photflag_detect : bitmask value for detections (from esr.photflag_detect)
    zp              : photometric zero-point
    zpsys           : zero-point system ('ab')
    band_prefix     : prefix to prepend to the band letter (e.g. 'lsst')

    Returns
    -------
    astropy.Table with columns: time, band, flux, fluxerr, zp, zpsys
    """
    df = ltcv_df.copy()

    # Keep only detected observations if requested
    if detected_only and photflag_detect is not None:
        df = df[(df['PHOTFLAG'] & photflag_detect) != 0]

    # Remove rows with non-positive flux error (bad measurements)
    df = df[df['FLUXCALERR'] > 0].copy()

    # Normalise band names: strip whitespace, lower-case, prepend prefix
    df['BAND'] = df['BAND'].str.strip().str.lower()
    df['band_sncosmo'] = band_prefix + df['BAND']

    table = astropy.table.Table({
        'time'    : df['MJD'].values.astype(float),
        'band'    : df['band_sncosmo'].values,
        'flux'    : df['FLUXCAL'].values.astype(float),
        'fluxerr' : df['FLUXCALERR'].values.astype(float),
        'zp'      : np.full(len(df), zp, dtype=float),
        'zpsys'   : np.full(len(df), zpsys),
    })
    return table


print("make_sncosmo_table helper ready.")

## 5 · Helper: fit a single event with SALT2

We use `sncosmo.fit_lc` which minimises chi-squared using the Minuit2 / iminuit backend
(or scipy if iminuit is not installed).  

**Free parameters**: `t0`, `x0`, `x1`, `c`.  
**Fixed parameter**: `z` — set to the truth redshift `ZCMB` from the ELAsTiCC2 catalogue.

You can optionally free `z` as well by adding it to `vparam_names`.

In [ ]:
def fit_salt2_event(ltcv_df: pd.DataFrame,
                    z_true: float,
                    fit_z: bool = False,
                    photflag_detect: int = None) -> dict:
    """Fit a single ELAsTiCC2 SNIa event with SALT2 using sncosmo.

    Parameters
    ----------
    ltcv_df         : DataFrame for a single SNID
    z_true          : truth redshift (ZCMB) used to initialise the model
    fit_z           : if True, redshift is also a free parameter
    photflag_detect : bitmask for detections

    Returns
    -------
    dict with keys:
        'success', 'result' (sncosmo result object), 'fitted_model',
        'table' (astropy table), 'z', 't0', 'x0', 'x1', 'c',
        'chi2', 'ndof', 'chi2_red', 'message'
    """
    # Build sncosmo observation table
    try:
        obs = make_sncosmo_table(
            ltcv_df,
            detected_only=DETECTED_ONLY,
            photflag_detect=photflag_detect
        )
    except Exception as e:
        return {'success': False, 'message': f'Table creation failed: {e}'}

    if len(obs) < 5:
        return {'success': False, 'message': 'Not enough data points after filtering'}

    # Initialise a fresh model instance
    model = sncosmo.Model(source=SALT2_SOURCE)

    # Initial parameter guess
    t0_guess = obs['time'][np.argmax(obs['flux'])]
    model.set(z=z_true, t0=t0_guess, x0=1e-4, x1=0.0, c=0.0)

    # Parameters to fit
    vparam_names = ['t0', 'x0', 'x1', 'c']
    bounds = {
        't0'  : (t0_guess - 30.0, t0_guess + 30.0),
        'x0'  : (1e-8, 1.0),
        'x1'  : (-5.0, 5.0),
        'c'   : (-0.5, 0.5),
    }
    if fit_z:
        vparam_names = ['z', 't0', 'x0', 'x1', 'c']
        bounds['z'] = (max(z_true - 0.1, 0.001), z_true + 0.1)

    # Run fit
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            result, fitted_model = sncosmo.fit_lc(
                obs, model,
                vparam_names=vparam_names,
                bounds=bounds,
                minsnr=0.0,       # include all points regardless of S/N
                warn=False
            )
    except Exception as e:
        return {'success': False, 'message': f'Fit failed: {e}', 'table': obs}

    chi2     = result.chisq
    ndof     = result.ndof
    chi2_red = chi2 / max(ndof, 1)

    return {
        'success'      : True,
        'result'       : result,
        'fitted_model' : fitted_model,
        'table'        : obs,
        'z'            : fitted_model['z'],
        't0'           : fitted_model['t0'],
        'x0'           : fitted_model['x0'],
        'x1'           : fitted_model['x1'],
        'c'            : fitted_model['c'],
        'chi2'         : chi2,
        'ndof'         : ndof,
        'chi2_red'     : chi2_red,
        'message'      : result.message,
    }


print("SALT2 single-event fitter ready.")

## 6 · Select events and run SALT2 fits

In [ ]:
# ── Sample N_CURVES events ─────────────────────────────────────────────────────
n_avail = min(N_CURVES, len(subset))
chosen_idx   = rng.choice(len(subset), size=n_avail, replace=False)
chosen_snids = subset['SNID'].values[chosen_idx]
print(f"Selected {n_avail} SNIDs for SALT2 fitting.")

In [ ]:
# ── Run SALT2 fits ─────────────────────────────────────────────────────────────
fit_results = {}

for snid in chosen_snids:
    ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]

    # True redshift from catalogue
    z_row = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    res = fit_salt2_event(
        ltcv,
        z_true=z_true,
        fit_z=False,
        photflag_detect=esr.photflag_detect
    )
    fit_results[snid] = res

    if res['success']:
        print(
            f"  SNID {snid:8d}  ✓  "
            f"z={res['z']:.4f}  t0={res['t0']:.2f}  "
            f"x0={res['x0']:.3e}  x1={res['x1']:+.3f}  c={res['c']:+.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}"
        )
    else:
        print(f"  SNID {snid:8d}  ✗  {res['message']}")

## 7 · Plot: multi-band light curves with SALT2 fit

For each event we show the raw flux data points (error bars) and the SALT2 model curve
in each band, using the band-specific colour defined in `BAND_COLORS`.  
**No renormalisation is applied**: the flux scale is absolute (FLUXCAL units).

In [ ]:
def plot_salt2_fit(ax, snid: int, res: dict, z_true: float):
    """Plot raw data + SALT2 model for one event on the given axes.

    Parameters
    ----------
    ax      : matplotlib Axes
    snid    : SNANA object identifier
    res     : dict returned by fit_salt2_event
    z_true  : truth redshift for the title
    """
    if not res.get('success'):
        ax.set_title(f"SNID {snid}\nFit failed\n{res.get('message','')}",
                     fontsize=8, color='red')
        return

    obs           = res['table']          # astropy.Table
    fitted_model  = res['fitted_model']   # sncosmo Model
    t0            = res['t0']
    chi2_red      = res['chi2_red']

    # Dense time axis centred on t0 for model curves
    t_min_data = obs['time'].min()
    t_max_data = obs['time'].max()
    t_dense = np.linspace(t_min_data - 10, t_max_data + 10, 400)

    for b in BANDS:
        bname = BAND_PREFIX + b
        color = BAND_COLORS.get(b, 'gray')

        # ── Data points for this band ──────────────────────────────────────────
        mask = np.array(obs['band']) == bname
        if mask.sum() > 0:
            t_data = obs['time'][mask]
            f_data = obs['flux'][mask]
            e_data = obs['fluxerr'][mask]
            ax.errorbar(
                t_data - t0, f_data, yerr=e_data,
                color=color, linestyle='None',
                marker='o', markersize=4, capsize=2,
                label=b, zorder=3
            )

        # ── SALT2 model curve for this band ────────────────────────────────────
        try:
            f_model = fitted_model.bandflux(
                bname, t_dense, zp=ZP, zpsys=ZPSYS
            )
            # Mask regions where model has no coverage (NaN or very negative)
            valid = np.isfinite(f_model)
            if valid.sum() > 1:
                ax.plot(
                    t_dense[valid] - t0, f_model[valid],
                    color=color, lw=1.5, ls='-', zorder=2
                )
        except Exception:
            pass

    ax.axhline(0.0, color='k', lw=0.5, ls='--')
    ax.set_title(
        f"SNID {snid}  z={z_true:.3f}\n"
        f"t0={t0:.1f}  x1={res['x1']:+.2f}  c={res['c']:+.2f}  "
        f"χ²/dof={chi2_red:.2f}",
        fontsize=8
    )
    ax.set_xlabel(r"$t - t_0$ [days]", fontsize=8)
    ax.set_ylabel("FLUXCAL", fontsize=8)
    ax.tick_params(axis='both', labelsize=7)
    ax.legend(fontsize=6, ncol=3, loc='upper right')


# ── Draw the grid of plots ─────────────────────────────────────────────────────
nrows = math.ceil(n_avail / NCOLS)
fig, axes = plt.subplots(
    nrows, NCOLS,
    figsize=(5.5 * NCOLS, 4.2 * nrows),
    tight_layout=True
)
axes_flat = np.array(axes).flatten()

for idx, snid in enumerate(chosen_snids):
    z_row = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan
    plot_salt2_fit(axes_flat[idx], snid, fit_results[snid], z_true)

# Hide unused subplots
for idx in range(n_avail, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(
    f"SALT2 fits — {OBJ_CLASS}  |  z ∈ [{Z_MIN},{Z_MAX})  |  ndet ≥ {MIN_DETECTIONS}\n"
    f"Model: {SALT2_SOURCE}  |  free params: t0, x0, x1, c  |  z fixed to truth",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

## 8 · Summary table of fitted parameters

In [ ]:
rows = []
for snid in chosen_snids:
    res = fit_results[snid]
    z_row = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    row = {
        'SNID'     : snid,
        'z_true'   : round(z_true, 4),
        'success'  : res.get('success', False),
    }
    if res.get('success'):
        row.update({
            'z_fit'    : round(res['z'],   4),
            't0'       : round(res['t0'],  2),
            'x0'       : float(f"{res['x0']:.4e}"),
            'x1'       : round(res['x1'],  3),
            'c'        : round(res['c'],   3),
            'chi2'     : round(res['chi2'], 2),
            'ndof'     : res['ndof'],
            'chi2_red' : round(res['chi2_red'], 3),
        })
    rows.append(row)

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

## 9 · Distribution of SALT2 parameters

In [ ]:
df_ok = df_results[df_results['success']].copy()

params = ['x1', 'c', 'chi2_red']
labels = {
    'x1'      : r'SALT2 $x_1$ (stretch)',
    'c'       : r'SALT2 $c$ (colour)',
    'chi2_red': r'$\chi^2$/dof'
}

fig2, axes2 = plt.subplots(1, len(params), figsize=(4.5 * len(params), 3.5),
                            tight_layout=True)

for ax, par in zip(axes2, params):
    vals = df_ok[par].dropna()
    ax.hist(vals, bins=10, color='steelblue', edgecolor='white')
    ax.axvline(np.median(vals), color='k', ls='--', lw=1.5,
               label=f'median={np.median(vals):.3g}')
    ax.set_xlabel(labels[par], fontsize=10)
    ax.set_ylabel('N events', fontsize=10)
    ax.set_title(f'σ = {np.std(vals):.3g}', fontsize=9)
    ax.legend(fontsize=9)

fig2.suptitle(
    f"SALT2 parameter distributions  —  {OBJ_CLASS}  (N={len(df_ok)} events)",
    fontsize=11
)
plt.show()

## 10 · Hubble diagram: distance modulus vs redshift

A quick Tripp-formula estimate of the distance modulus:

$$
\mu = m_B^* - M_B + \alpha\, x_1 - \beta\, c
$$

with standard values $\alpha = 0.14$, $\beta = 3.1$.  
The apparent magnitude $m_B^*$ is derived from $x_0$ via $m_B^* = -2.5\log_{10}(x_0) + \text{const}$,
where the SALT2 normalisation convention gives $\text{const} = 10.635$ for the `salt2-extended` model.

In [ ]:
# SALT2 standard candle parameters (Betoule et al. 2014 values)
ALPHA_TRIPP = 0.14
BETA_TRIPP  = 3.10
# salt2-extended: mB* = -2.5 log10(x0) + 10.635  (model-dependent constant)
MB_CONST    = 10.635
# SDC empirical correction to the formula
MB_CORR_SDC = +15.

df_ok = df_ok.copy()
df_ok['mB_star'] = -2.5 * np.log10(df_ok['x0'].astype(float)) + MB_CONST
df_ok['mu_tripp'] = (df_ok['mB_star']
                     + ALPHA_TRIPP * df_ok['x1']
                     - BETA_TRIPP  * df_ok['c'] + MB_CORR_SDC)

fig3, ax3 = plt.subplots(figsize=(7, 4.5), tight_layout=True)
sc = ax3.scatter(
    df_ok['z_true'], df_ok['mu_tripp'],
    c=df_ok['chi2_red'], cmap='viridis_r',
    vmin=0, vmax=5,
    s=40, edgecolors='k', linewidths=0.3, zorder=3
)
plt.colorbar(sc, ax=ax3, label=r'$\chi^2$/dof')

# Flat ΛCDM reference (simplified: mu ≈ 5 log10(d_L/10pc))
try:
    from astropy.cosmology import FlatLambdaCDM
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    z_ref = np.linspace(Z_MIN, Z_MAX, 200)
    mu_ref = cosmo.distmod(z_ref).value
    ax3.plot(z_ref, mu_ref, 'r--', lw=1.5, label=r'Flat ΛCDM ($H_0$=70, $\Omega_m$=0.3)')
    ax3.legend(fontsize=9)
except Exception:
    pass

ax3.set_xlabel('Redshift z (truth)', fontsize=11)
ax3.set_ylabel(r'$\mu_{\rm Tripp}$ (relative)', fontsize=11)
ax3.set_title(
    f"Hubble diagram — {OBJ_CLASS}  ({len(df_ok)} events)\n"
    rf"$\mu = m_B^* + {ALPHA_TRIPP}\,x_1 - {BETA_TRIPP}\,c$",
    fontsize=11
)
plt.show()

## Summary

| Parameter | Status | Description |
|-----------|--------|-------------|
| `z`  | fixed | truth redshift from ELAsTiCC2 catalogue |
| `t0` | free  | time of B-band maximum (MJD) |
| `x0` | free  | overall amplitude (proportional to luminosity) |
| `x1` | free  | SALT2 stretch (light-curve width) |
| `c`  | free  | SALT2 colour (SED tilt) |

All six LSST bands (u, g, r, i, z, y) are fitted **simultaneously** with a single
set of parameters. The SALT2 spectral template handles the multi-band flux ratios
intrinsically — no per-band renormalisation or colour factors are needed.

The fit is performed by `sncosmo.fit_lc`, which minimises the chi-squared between
the observed `FLUXCAL` values and the model `bandflux` predictions at zero-point
`ZP=31.4` (AB system).
